In [1]:
from dotenv import load_dotenv
load_dotenv()  # 自动寻找 .env 文件并加载到 os.environ

True

### 配置GPU加速处理

In [2]:
import os
os.environ["TORCH_DEVICE"] = "cuda"
os.environ["INFERENCE_RAM"] = "6"  # 你的显卡是 6GB

## 运行marker进行少量pdf解析

In [6]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser

# ------------------ 配置 ------------------
INPUT_PDF_PATH = r"knowledgeBase\pdfParsed_input\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library).pdf"

config = {
    "output_format": "markdown",
    "paginate_output": True,
    "force_ocr":True,
    "DocumentBuilder_lowres_image_dpi": 150,
    "TORCH_DEVICE": "cuda",  # 明确指定
    "inference_ram": 6,     # 明确指定显存
    "batch_multiplier": 2,  
    "page_range":"",
    # "debug":True,
}
model_dict = create_model_dict()
config_parser = ConfigParser(config)

converter = PdfConverter(
    config = config_parser.generate_config_dict(),
    artifact_dict = model_dict,
)

rendered = converter(INPUT_PDF_PATH)
text, _, images = text_from_rendered(rendered)
print("转换完成")


Recognizing Text: 100%|██████████| 13/13 [01:59<00:00,  9.19s/it]
Detecting bboxes: 0it [00:00, ?it/s]

转换完成


In [7]:
print(text)



{19}------------------------------------------------

度之严肃认真,工作作风之深入细致,也是令人钦佩的。在这样的基础上产生的营造法式,何愁不能广泛施行?如果说李诫这部巨著在很大程度上反映了北宋建筑工程的成就和匠师的智慧与经验,恐怕也并不为过。

正由于李诫创立了合理的体例和密切结合实际的工作路线,所以此书编成后能顺利得到主管部门认可,在京师地区推出试行。三年之后,李诫又提出要在京师以外地区推广,用小字刻版刊印,作为朝廷敕命通行的文本发至各地遵照执行。这个请求得到宋徽宗的批准,于崇宁二年(1103年)刊印了这部《法式》。直到宋室南迁,政治中心易地,平江知府王晚还在苏州重新刻印此书,以应工程之需,南宋后期,又在平江重印了一次,这也从实践方面证明了此书的广泛适用性(详见本书 232 页附录一"宋版书"条)。

## 二、《营造法式》与江南建筑的关系

浙江宁波保国寺大殿是北宋前期所建的一座木架建筑<sup>(n)</sup>,它的四根内柱是用拼合法制成的,其中三根是由八根木料拼成的八瓣形柱子,一根是由九根木料拼成的八瓣柱子 <sup>(s)</sup>(图 1-1)。这不仅是国内已知最早的拼合柱实例,也是宋代拼柱实物的孤例。而在此殿建成后九十余年问世的《营造法式》中也载有拼柱法,虽然拼合方法不同,但同是用小料拼成大部件以解决用料的困难(参见图 2-43)。多年前我们为了苏州瑞光寺塔的修复设计<sup>(s)</sup>,曾去杭州考察了五代末年至宋初的几座石塔和经幢<sup>(n)</sup>,联系江苏宝应出土的南唐木屋<sup>(n)</sup>和江苏镇江甘露寺铁塔第一、二层来考虑<sup>(n)</sup>(图 1-2~1-5),感到五代至北宋间江南一带的建筑和《营造法式》的做法很接近,尤其是大木作,几座石塔的斗栱、柱、枋、檐部等,几乎都可和《法式》相印证,屋角起翘也较平缓,和明、清时期江南的"嫩戗发戗"迥然不同<sup>(n)</sup>。因而我们联想到北宋初年曾在汴京名噪一时的建筑大师喻皓,他从杭州去京师后,于端拱二年(989年)也就是杭州灵隐寺石塔建成后 29年,在东京建成了著名的开宝寺十一层木塔,他著的《木经》则被奉为营造典范流行于世,是李诫《法式》问世前的权威性建筑著作。所以说由于喻皓的实践和《木经》的

## 保存本次pdf解析结果

In [ ]:
import os
from datetime import datetime

output_dir = r"knowledgeBase\pdfParsed_output"
os.makedirs(output_dir, exist_ok=True)

# ===== 2. 获取输入文件名（不含后缀）=====
input_filename = os.path.splitext(os.path.basename(INPUT_PDF_PATH))[0]
# ===== 3. 生成时间戳，避免覆盖 =====
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# ===== 4. 保存 Markdown 结果 =====
text_output_path = os.path.join(
    output_dir,
    f"{input_filename}_result_{timestamp}.md"
)

with open(text_output_path, "w", encoding="utf-8") as f:
    f.write(text)

# ===== 5. 保存图片（如果有）=====
image_output_dir = os.path.join(
    output_dir,
    f"{input_filename}_images_{timestamp}"
)
saved_images = []

if images:
    os.makedirs(image_output_dir, exist_ok=True)
    for idx, (image_name, image_data) in enumerate(images.items()):
        image_path = os.path.join(image_output_dir, image_name)
        # image_data 可能是 PIL Image 或 bytes
        if hasattr(image_data, "save"):  # PIL Image
            image_data.save(image_path)
        else:  # bytes
            with open(image_path, "wb") as img_f:
                img_f.write(image_data)
        saved_images.append(image_path)

# ===== 6. Notebook 输出说明 =====
print("✅ PDF 解析结果已保存")
print(f"📄 文本/markdown 文件: {text_output_path}")

if saved_images:
    print(f"图片目录: {image_output_dir}")
    print(f"图片数量: {len(saved_images)}")
else:
    print("本次解析未产生可导出的图片")


✅ PDF 解析结果已保存
📄 文本/markdown 文件: knowledgeBase\pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_result_20260203_095333.md
图片目录: knowledgeBase\pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_images_20260203_095333
图片数量: 4


## 解析整本pdf 300+页数

In [1]:
import os
import torch
import gc
from datetime import datetime
import pypdf
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser

def get_pdf_total_pages(pdf_path):
    """获取PDF文件的总页数"""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = pypdf.PdfReader(file)
            return len(pdf_reader.pages)
    except Exception as e:
        print(f"无法读取PDF页数: {e}")
        return 0

def batch_process_pdf(
    pdf_path, 
    output_dir, 
    batch_size=30, 
    start_batch_name_nummber=1, 
    start_page=None,
    end_page=None
):
    """
    分批处理大型PDF，支持中断后从指定批次/页码继续处理
    Args:
        pdf_path: PDF文件路径
        output_dir: 输出目录
        batch_size: 每批处理的页数（默认30）
        start_batch: 起始批次号（仅用于文件命名，从1开始，默认1）
        start_page: 起始页码（从1开始，可选）
        end_page: 结束页码（从1开始，可选）
    """
    # 1. 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 获取输入文件名
    input_filename = os.path.splitext(os.path.basename(pdf_path))[0]
    
    # 2. 检测PDF总页数
    total_pages = get_pdf_total_pages(pdf_path)
    if total_pages == 0:
        print(f"错误: 无法读取PDF文件: {pdf_path}")
        return None
    
    print(f"📄 PDF文件: {input_filename}")
    print(f"📊 总页数: {total_pages}")
    
    # 3. 确定实际处理范围
    # 起始页码，默认从第1页开始
    if start_page is not None:
        actual_start_page = max(1, min(start_page, total_pages))
    else:
        actual_start_page = 1
    
    # 结束页码，默认到最后一页
    if end_page is not None:
        actual_end_page = min(end_page, total_pages)
    else:
        actual_end_page = total_pages
    
    # 验证处理范围
    if actual_start_page > actual_end_page:
        print(f"错误: 起始页码({actual_start_page})大于结束页码({actual_end_page})")
        return None
    
    print(f"⚙️  处理参数:")
    print(f"  批次大小: {batch_size} 页")
    print(f"  页码范围: {actual_start_page}-{actual_end_page}")
    
    # 4. 检查设备
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  处理设备: {device}")
    
    # 5. 创建模型字典（共享使用）
    print("🔧 加载模型...")
    try:
        artifact_dict = create_model_dict()
        print("✅ 模型加载完成")
    except Exception as e:
        print(f"❌ 模型加载失败: {e}")
        return None
    
    # 6. 计算总批次数
    total_pages_to_process = actual_end_page - actual_start_page + 1
    total_batches = (total_pages_to_process + batch_size - 1) // batch_size
    
    print(f"📈 需要处理: {total_pages_to_process}页, {total_batches}个批次")
    print(f"  起始批次号: {start_batch_name_nummber} (仅用于文件命名)")

    
    # 7. 分批处理PDF
    batch_results = []
    
    for batch_idx in range(total_batches):
        current_batch = start_batch_name_nummber + batch_idx
        
        # 计算当前批次的页码范围
        batch_start_page = actual_start_page + batch_idx * batch_size
        batch_end_page = min(batch_start_page + batch_size - 1, actual_end_page)
        page_range = f"{batch_start_page}-{batch_end_page}"
        
        print(f"\n{'='*60}")
        print(f"🔄 正在处理批次 {current_batch}: 第 {page_range} 页")
        print(f"{'='*60}")
        
        try:
            # 创建批次配置
            config = {
                "output_format": "markdown",
                "paginate_output": True,
                "force_ocr": True,
                "DocumentBuilder_lowres_image_dpi": 150,
                "TORCH_DEVICE": device,
                "inference_ram": 2,
                "batch_multiplier": 1,
                "page_range": page_range,
                "max_pages": batch_size,
            }
            
            # 创建配置解析器和转换器
            config_parser = ConfigParser(config)
            converter = PdfConverter(
                config=config_parser.generate_config_dict(),
                artifact_dict=artifact_dict,
            )
            
            # 处理当前批次
            rendered = converter(pdf_path)
            text, _, images = text_from_rendered(rendered)
            
            # 统计信息
            text_length = len(text)
            image_count = len(images) if images else 0
            
            print(f"✅ 批次 {current_batch} 处理完成:")
            print(f"   页数范围: {page_range}")
            print(f"   文本长度: {text_length:,} 字符")
            print(f"   图片数量: {image_count}")
            
            # 保存当前批次结果到文件
            batch_output_path = os.path.join(
                output_dir,
                f"{input_filename}_batch_{current_batch:03d}_{batch_start_page}-{batch_end_page}.md"
            )
            
            with open(batch_output_path, "w", encoding="utf-8") as f:
                f.write(f"# 批次 {current_batch}: 第 {page_range} 页\n\n")
                f.write(f"处理时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                f.write(f"总字符数: {text_length}\n\n")
                f.write("---\n\n")
                f.write(text)
            
            print(f"💾 批次结果已保存: {batch_output_path}")
            
            # 保存批次图片（如果存在）
            if images:
                batch_image_dir = os.path.join(
                    output_dir,
                    f"{input_filename}_batch_{current_batch:03d}_{batch_start_page}-{batch_end_page}_images"
                )
                os.makedirs(batch_image_dir, exist_ok=True)
                
                saved_count = 0
                for image_name, image_data in images.items():
                    # 确保文件名安全
                    safe_image_name = "".join(c for c in image_name if c.isalnum() or c in '._- ').rstrip()
                    if not safe_image_name:
                        safe_image_name = f"image_{saved_count+1:03d}.png"
                    
                    image_path = os.path.join(batch_image_dir, safe_image_name)
                    
                    try:
                        if hasattr(image_data, "save"):  # PIL Image
                            # 确保有正确的文件扩展名
                            if not any(safe_image_name.lower().endswith(ext) for ext in ['.png', '.jpg', '.jpeg', '.gif', '.bmp']):
                                image_path += '.png'
                            image_data.save(image_path)
                        else:  # bytes
                            with open(image_path, "wb") as img_f:
                                img_f.write(image_data)
                        
                        saved_count += 1
                    except Exception as img_e:
                        print(f"警告: 无法保存图片 {image_name}: {img_e}")
                
                print(f"🖼️  批次图片已保存: {saved_count} 张 ({batch_image_dir})")
            else:
                print("ℹ️  本批次未发现图片")
            
            # 存储批次结果信息
            batch_results.append({
                "batch_id": current_batch,
                "page_range": page_range,
                "text_length": text_length,
                "image_count": image_count,
                "file_path": batch_output_path,
                "status": "processed"
            })
            
            # 清理内存
            del converter, rendered
            if device == "cuda":
                torch.cuda.empty_cache()
            gc.collect()
            
        except Exception as e:
            print(f"❌ 批次 {current_batch} 处理失败: {e}")
            import traceback
            traceback.print_exc()
            
            batch_results.append({
                "batch_id": current_batch,
                "page_range": page_range,
                "error": str(e),
                "status": "failed"
            })
            continue
    
    # 8. 生成处理报告
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    report_path = os.path.join(output_dir, f"processing_report_{timestamp}.txt")
    
    processed_count = len([b for b in batch_results if b.get("status") == "processed"])
    failed_count = len([b for b in batch_results if b.get("status") == "failed"])
    
    with open(report_path, "w", encoding="utf-8") as report:
        report.write(f"PDF处理报告\n")
        report.write(f"{'='*40}\n")
        report.write(f"处理时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        report.write(f"输入文件: {pdf_path}\n")
        report.write(f"总页数: {total_pages}\n")
        report.write(f"处理设备: {device}\n")
        report.write(f"批次大小: {batch_size}\n")
        report.write(f"起始批次号: {start_batch_name_nummber}\n")
        report.write(f"处理范围: {actual_start_page}-{actual_end_page}\n\n")
        
        report.write(f"处理统计:\n")
        report.write(f"  总批次: {len(batch_results)}\n")
        report.write(f"  成功处理: {processed_count}\n")
        report.write(f"  失败: {failed_count}\n\n")
        
        if processed_count > 0:
            report.write("成功处理的批次:\n")
            total_text_length = 0
            total_images = 0
            
            for batch in batch_results:
                if batch.get("status") == "processed":
                    report.write(f"  批次 {batch['batch_id']}: {batch['page_range']}页, "
                                f"{batch['text_length']}字符, {batch['image_count']}图片\n")
                    total_text_length += batch.get("text_length", 0)
                    total_images += batch.get("image_count", 0)
            
            report.write(f"\n总计: {total_text_length}字符, {total_images}张图片\n")
        
        if failed_count > 0:
            report.write(f"\n失败的批次:\n")
            for batch in batch_results:
                if batch.get("status") == "failed":
                    report.write(f"  批次 {batch['batch_id']}: {batch['page_range']}页 - {batch.get('error', '未知错误')}\n")
    
    # 9. 输出处理摘要
    print(f"\n{'='*60}")
    print("✅ PDF处理完成!")
    print(f"{'='*60}")
    print(f"📄 输入文件: {input_filename}")
    print(f"📊 处理范围: {actual_start_page}-{actual_end_page}页")
    print(f"🔢 总批次: {len(batch_results)}")
    print(f"✅ 成功处理: {processed_count}")
    print(f"❌ 失败: {failed_count}")
    print(f"📋 处理报告: {report_path}")
    print(f"📁 输出目录: {output_dir}")
    
    return {
        "batch_results": batch_results,
        "report_path": report_path,
        "total_batches": len(batch_results),
        "processed": processed_count,
        "failed": failed_count
    }

# 使用示例
if __name__ == "__main__":
    # 方式1: 直接指定参数
    INPUT_PDF_PATH = r"knowledgeBase\pdfParsed_input\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library).pdf"
    OUTPUT_DIR = r"knowledgeBase/pdfParsed_output"
    result = batch_process_pdf(
        pdf_path = INPUT_PDF_PATH,
        output_dir = OUTPUT_DIR,
        batch_size = 25,           # 每批25页
        start_batch_name_nummber = 7,           # 文件命名从第7批开始
        start_page = 55,           # 从第55页开始处理
        end_page = 323             # 处理到第323页
    )
    if result:
        print("\n🎉 处理完成!")
    else:
        print("\n❌ 处理失败!")

📄 PDF文件: 《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)
📊 总页数: 340
⚙️  处理参数:
  批次大小: 25 页
  页码范围: 55-323
  处理设备: cuda
🔧 加载模型...
✅ 模型加载完成
📈 需要处理: 269页, 11个批次
  起始批次号: 7 (仅用于文件命名)

🔄 正在处理批次 7: 第 55-79 页


Recognizing Text: 100%|██████████| 225/225 [02:09<00:00,  1.74it/s]


✅ 批次 7 处理完成:
   页数范围: 55-79
   文本长度: 29,665 字符
   图片数量: 17
💾 批次结果已保存: knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_007_55-79.md
🖼️  批次图片已保存: 17 张 (knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_007_55-79_images)

🔄 正在处理批次 8: 第 80-104 页


Recognizing Text: 100%|██████████| 80/80 [00:58<00:00,  1.37it/s]


✅ 批次 8 处理完成:
   页数范围: 80-104
   文本长度: 22,007 字符
   图片数量: 44
💾 批次结果已保存: knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_008_80-104.md
🖼️  批次图片已保存: 44 张 (knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_008_80-104_images)

🔄 正在处理批次 9: 第 105-129 页


...

✅ 批次 15 处理完成:
   页数范围: 255-279
   文本长度: 16,935 字符
   图片数量: 0
💾 批次结果已保存: knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_015_255-279.md
ℹ️  本批次未发现图片

🔄 正在处理批次 16: 第 280-304 页


Recognizing Text: 100%|██████████| 617/617 [04:58<00:00,  2.06it/s]


✅ 批次 16 处理完成:
   页数范围: 280-304
   文本长度: 35,229 字符
   图片数量: 10
💾 批次结果已保存: knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_016_280-304.md
🖼️  批次图片已保存: 10 张 (knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_016_280-304_images)

🔄 正在处理批次 17: 第 305-323 页


Recognizing Text: 100%|██████████| 560/560 [03:26<00:00,  2.71it/s]


✅ 批次 17 处理完成:
   页数范围: 305-323
   文本长度: 26,962 字符
   图片数量: 12
💾 批次结果已保存: knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_017_305-323.md
🖼️  批次图片已保存: 12 张 (knowledgeBase/pdfParsed_output\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)_batch_017_305-323_images)

✅ PDF处理完成!
📄 输入文件: 《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library)
📊 处理范围: 55-323页
🔢 总批次: 11
✅ 成功处理: 11
❌ 失败: 0
📋 处理报告: knowledgeBase/pdfParsed_output\processing_report_20260203_194904.txt
📁 输出目录: knowledgeBase/pdfParsed_output

🎉 处理完成!


## 解析pdf的书签数据作为精确目录

In [3]:
import fitz  # PyMuPDF
import os
import json

def export_pdf_bookmarks_to_jsonl(pdf_path):
    """
    提取PDF书签数据并导出为JSONL格式
    
    参数:
        pdf_path: PDF文件路径
    
    输出:
        生成与PDF同名的_jsonl文件，每行一个JSON对象
        包含三个字段：章节标题(title)、层级(level)、页码(page)
    """
    # 1. 检查文件是否存在
    if not os.path.exists(pdf_path):
        print(f"错误：找不到文件 '{pdf_path}'，请检查路径。")
        return False

    try:
        # 2. 打开PDF文件
        doc = fitz.open(pdf_path)
        toc = doc.get_toc()  # 获取原始书签数据
        
        if not toc:
            print("提示：该PDF文件内部没有书签数据。")
            return False

        # 3. 准备输出文件名（JSONL格式）
        output_filename = os.path.splitext(pdf_path)[0] + "_bookmarks.jsonl"
        
        print(f"\n正在处理: {os.path.basename(pdf_path)}")
        print("-" * 40)
        
        # 4. 写入JSONL文件
        with open(output_filename, "w", encoding="utf-8") as f:
            for entry in toc:
                level, title, page = entry[0], entry[1], entry[2]
                
                # 构建JSON对象
                bookmark_item = {
                    "章节标题": title,
                    "层级": level,
                    "页码": page
                }
                
                # 写入JSONL文件（一行一个JSON）
                f.write(json.dumps(bookmark_item, ensure_ascii=False) + "\n")
                
                # 控制台输出（可选，便于调试）
                indent = "  " * (level - 1)
                print(f"{indent}L{level} P{page:<4} {title}")

        print("-" * 40)
        print(f"✅ 成功导出书签数据到：")
        print(f"👉 {os.path.abspath(output_filename)}")
        print(f"📊 共导出 {len(toc)} 个目录项")
        
        return True

    except Exception as e:
        print(f"运行出错：{e}")
        return False
    finally:
        if 'doc' in locals():
            doc.close()

# --- 执行部分 ---
if __name__ == "__main__":
    # 替换为你的PDF文件路径
    pdf_file = r"knowledgeBase\pdfParsed_input\《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library).pdf"
    
    # 导出书签到JSONL文件
    export_pdf_bookmarks_to_jsonl(pdf_file)


正在处理: 《营造法式》解读 第2版 (潘谷西；何建中著) (Z-Library).pdf
----------------------------------------
L1 P1    封面
L1 P3    书名
L1 P4    版权
L1 P5    前言
L1 P8    目录
L1 P16   第一章  总论
  L2 P16   一、《营造法式》的性质与特点
    L3 P16   （一）“营造法式”是一种建筑工程预算定额
    L3 P17   （二）李诫《营造法式》的编写体例
    L3 P19   （三）编写工作紧密结合实际
  L2 P20   二、《营造法式》与江南建筑的关系
  L2 P29   三、《营造法式》的内容取舍
  L2 P31   四、宋代官式建筑分类
  L2 P32   五、研究方法的讨论
  L2 P34   第一章注释
L1 P36   第二章  木构架
  L2 P36   一、宋代官式建筑木构架的基本类型
    L3 P36   （一）柱梁作
    L3 P38   （二）殿阁式木构架
      L4 P38   1.柱框层
      L4 P42   2.铺作层
      L4 P43   3.屋盖层
    L3 P44   （三）厅堂式木构架
    L3 P56   （四）楼阁木构架
      L4 P56   1.层叠构架楼阁
      L4 P56   2.混合整体构架楼阁
  L2 P58   二、材、栔、分°模数
  L2 P61   三、定“地盘”——木构架的平面设定
  L2 P63   四、定“侧样”——木构架的剖面设定
    L3 P63   （一）下份
    L3 P65   （二）中份
      L4 P65   1.柱高
      L4 P65   2.铺作高
    L3 P67   （三）上份
      L4 P67   1.架深
      L4 P67   2.檐出
      L4 P68   3.举折
  L2 P69   五、再谈定“侧样”中的架深
  L2 P72   六、定“正样”——木构架的立面设定
    L3 P72   （一）影响房屋立面的八种因素
    L3 P74   （二）关于间广和柱高的讨论
    L3 P75 